# Figure 2 — admixture mapping


In [ ]:
import sys, os, pathlib
os.environ['MPLBACKEND'] = 'Agg'
import matplotlib
matplotlib.use('Agg', force=True)
import matplotlib.pyplot as plt
plt.ioff()

try:
    _ip = get_ipython()
    _ip.run_line_magic('load_ext', 'autoreload')
    _ip.run_line_magic('autoreload', '2')
except Exception:
    pass

try:
    _cwd = pathlib.Path.cwd()
except OSError:
    os.chdir(os.path.expanduser('~'))
    _cwd = pathlib.Path.cwd()

def _is_root(p):
    return (p / 'scripts').is_dir() and (p / '.git').exists()

_home = pathlib.Path.home()
_candidates = [_cwd, pathlib.Path(os.environ.get('PWD', _cwd)),
               _home, _home / 'SelectionMapping']
_root = None
for _base in _candidates:
    for _p in (_base, *_base.parents):
        if _is_root(_p):
            _root = _p; break
    if _root:
        break
_root = _root or _cwd
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import importlib
for _name in [m for m in list(sys.modules) if m == 'scripts' or m.startswith('scripts.')]:
    try:
        importlib.reload(sys.modules[_name])
    except Exception:
        pass

from scripts import figure2_data, figure2_plots, figure2_loci
from scripts.common import panel_io, style
style.apply_style()


## Run or resume the analysis and build Figure 2


In [ ]:
clean_run_data = False
clean_run_plots = True

locus_status = figure2_data.prepare_locus_outputs(force=clean_run_data)
print('[figure2_sources] locus preparation:', locus_status)

fig = figure2_plots.build_figure2_sources(
    clean_run_data=clean_run_data,
    clean_run_plots=clean_run_plots,
)
plt.close("all")

from scripts import figure2_plrs_supplement
plrs_supplement = figure2_plrs_supplement.run(force=clean_run_plots)
print("[figure2_sources] PLRS supplements:", plrs_supplement)
nominal_loci = figure2_loci.run(force=clean_run_data)
print('[figure2_sources] nominal discovery loci:', nominal_loci)
plt.close('all')


## Saved files (text only)


In [ ]:
for _p in panel_io.panel_image_paths('figure2_sources'):
    if _p.exists():
        print(_p)
for _p in figure2_plots.supplemental_tract_paths():
    if _p.suffix == '.png' and _p.exists():
        print(_p)
if globals().get('nominal_loci', {}).get('status') == 'ready':
    _locus_pngs = sorted(figure2_loci.output_dir().glob('*.png'))
    print(f'Saved nominal loci: {figure2_loci.output_dir()} ({len(_locus_pngs)} PNGs)')
    for _p in _locus_pngs:
        print(_p)


Outputs written:

- Main panels: `figures/figure2_sources/panels/Figure_2<letter>.{pdf,png}`
- Supplemental tract panel: `figures/figure2_sources/panels/Figure_2_admixture_tracts_supplemental.{pdf,png}`
- Composite: `figures/figure2_sources/Figure_2.{pdf,png}`
- Reused analysis intermediates: `analysis/figure4/`
- All corrected discovery-nominal locus pairs: `figures/figure2_sources/loci/chr<chrom>_<start>_<end>__<phecode>.{pdf,png}`
- Locus summary and provenance: `figures/figure2_sources/loci/fine_mapping_summary.tsv` and `run_manifest.json`

The additional locus output reads `figures/figure2_sources/Figure_2_1025_phewas_results.csv` and the matching corrected caches under `analysis/figure2/`. It includes every association with discovery p < 0.001 (currently 13), irrespective of replication availability or meta significance. Each uses the 10k discovery cohort and the Figure 2f region-boundary +/-100 kb SuSiE method. Existing Figure 2f fits are reused when their sample/parameter/output manifests validate. New fits are checkpointed individually, and failed fits make the run fail explicitly. Plots include PIP, score-test association, LD colour, SV markers and exon/intron tracks; PDFs have editable text and no raster layers.